# HEEDB data description table

The HEEDB table from `results/data-tables.ipynb`, on the by-label splits with blank overreads dropped, with labels from `label_src`: `audited_physician` (default) or `original_physician`. The audited source only counts codes in the main read, not the comparison text (see `protossl/datasets/_heedb_label_rules.py`). Label rows count positives as a % of the split; `audited_physician` unknowns (`UNKNOWN_LABEL`) count as not positive here and are broken down in `heedb-muse-overread-table.ipynb`. With `original_physician` the table matches the one in `results/data-tables.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
from typing import Optional

from protossl.datasets import HeedbECGDataset
from protossl.defines import HEEDB_TARGETS


class TableOne:
    """
    A simplified, fast version of TableOne for large datasets.

    Parameters
    ----------
    data : pd.DataFrame
        Input data.
    columns : list
        Columns to include in the table.
    categorical : list
        Categorical columns.
    continuous : list
        Continuous columns.
    nonnormal : list, optional
        Continuous columns to display as median [Q1, Q3] instead of mean (SD).
    nunique : list, optional
        Columns for which to report the number of unique values per group
        (e.g. patient ID, study ID). Rendered as a single row per column.
    groupby : str, optional
        Column to group by (becomes column headers).
    order : dict, optional
        Mapping of column name -> ordered list of values. Used for groupby column
        ordering and for ordering categorical row values.
    binary_show : dict, optional
        Mapping of binary categorical column name -> the single class to display.
        Only columns explicitly listed here will be collapsed to a single row;
        binary columns not listed are rendered with all levels like any other
        categorical.
    """

    def __init__(
        self,
        data: pd.DataFrame,
        columns: list,
        categorical: Optional[list] = None,
        continuous: Optional[list] = None,
        nonnormal: Optional[list] = None,
        nunique: Optional[list] = None,
        groupby: Optional[str] = None,
        order: Optional[dict] = None,
        binary_show: Optional[dict] = None,
    ):
        self.data = data
        self.columns = list(columns)
        self.categorical = list(categorical) if categorical else []
        self.continuous = list(continuous) if continuous else []
        self.nonnormal = set(nonnormal) if nonnormal else set()
        self.nunique = list(nunique) if nunique else []
        self.groupby = groupby
        self.order = order or {}
        self.binary_show = binary_show or {}

        # Don't render the groupby column as a row — it's a diagonal submatrix.
        if self.groupby is not None:
            self.categorical = [c for c in self.categorical if c != self.groupby]
            self.continuous = [c for c in self.continuous if c != self.groupby]
            self.nunique = [c for c in self.nunique if c != self.groupby]
            self.columns = [c for c in self.columns if c != self.groupby]

        # Make sure nunique columns appear in self.columns even if the user
        # didn't list them explicitly.
        for c in self.nunique:
            if c not in self.columns:
                self.columns.append(c)

        self.table = self._build()

    # ------------------------------------------------------------------ #
    # Group setup
    # ------------------------------------------------------------------ #
    def _group_indexers(self):
        """Return list of (group_label, boolean_mask, n) tuples."""
        if self.groupby is None:
            mask = np.ones(len(self.data), dtype=bool)
            return [("Overall", mask, int(mask.sum()))]

        col = self.data[self.groupby]
        if self.groupby in self.order:
            levels = list(self.order[self.groupby])
        else:
            levels = list(pd.unique(col.dropna()))

        out = []
        for lvl in levels:
            mask = (col == lvl).to_numpy()
            out.append((lvl, mask, int(mask.sum())))
        return out

    # ------------------------------------------------------------------ #
    # Continuous formatting
    # ------------------------------------------------------------------ #
    @staticmethod
    def _fmt_mean_sd(x: np.ndarray) -> str:
        x = x[~np.isnan(x)]
        if x.size == 0:
            return ""
        return f"{x.mean():.1f} ({x.std(ddof=1):.1f})"

    @staticmethod
    def _fmt_median_iqr(x: np.ndarray) -> str:
        x = x[~np.isnan(x)]
        if x.size == 0:
            return ""
        q1, med, q3 = np.quantile(x, [0.25, 0.5, 0.75])
        return f"{med:.1f} [{q1:.1f}, {q3:.1f}]"

    def _continuous_row(self, col: str, groups):
        nonnormal = col in self.nonnormal
        label = f"{col}, {'median [Q1, Q3]' if nonnormal else 'mean (SD)'}"
        values = self.data[col].to_numpy(dtype=float, na_value=np.nan)

        row = {}
        for gname, mask, _ in groups:
            sub = values[mask]
            row[gname] = self._fmt_median_iqr(sub) if nonnormal else self._fmt_mean_sd(sub)
        return label, row

    # ------------------------------------------------------------------ #
    # Unique-count formatting
    # ------------------------------------------------------------------ #
    def _nunique_row(self, col: str, groups):
        label = f"{col}, unique n"
        series = self.data[col].to_numpy()
        row = {}
        for gname, mask, _ in groups:
            sub = series[mask]
            # pd.unique handles NaN, mixed types, and is faster than np.unique
            # on object dtype.
            row[gname] = f"{pd.unique(sub).size}"
        return label, row

    # ------------------------------------------------------------------ #
    # Categorical formatting
    # ------------------------------------------------------------------ #
    def _categorical_rows(self, col: str, groups):
        """
        Yields (label, {group: 'n (pct%)'} ) tuples.
        Only collapses to a single row if the user has explicitly listed the
        column in binary_show.
        """
        series = self.data[col]

        # Explicit single-class selection — only path that collapses the column.
        if col in self.binary_show:
            chosen = self.binary_show[col]
            label = f"{col} = {chosen}, n (%)"
            row = {}
            chosen_mask = (series == chosen).to_numpy()
            for gname, mask, n in groups:
                if n == 0:
                    row[gname] = ""
                    continue
                count = int((chosen_mask & mask).sum())
                pct = 100.0 * count / n
                row[gname] = f"{count} ({pct:.1f})"
            yield label, row
            return

        # Determine level order
        if col in self.order:
            levels = list(self.order[col])
        else:
            levels = sorted(series.dropna().unique().tolist(), key=lambda v: (str(type(v)), v))

        # Header row + one row per level
        yield f"{col}, n (%)", {gname: "" for gname, _, _ in groups}
        for lvl in levels:
            label = f"    {lvl}"
            row = {}
            lvl_mask = (series == lvl).to_numpy()
            for gname, mask, n in groups:
                if n == 0:
                    row[gname] = ""
                    continue
                count = int((lvl_mask & mask).sum())
                pct = 100.0 * count / n
                row[gname] = f"{count} ({pct:.1f})"
            yield label, row

    # ------------------------------------------------------------------ #
    # Build
    # ------------------------------------------------------------------ #
    def _build(self) -> pd.DataFrame:
        groups = self._group_indexers()
        group_names = [g[0] for g in groups]

        rows = []
        index = []

        # n row
        index.append("n")
        rows.append({gname: f"{n}" for gname, _, n in groups})

        nunique_set = set(self.nunique)

        # Preserve user-specified column order
        for col in self.columns:
            if col in nunique_set:
                label, row = self._nunique_row(col, groups)
                index.append(label)
                rows.append(row)
            elif col in self.continuous:
                label, row = self._continuous_row(col, groups)
                index.append(label)
                rows.append(row)
            elif col in self.categorical:
                for label, row in self._categorical_rows(col, groups):
                    index.append(label)
                    rows.append(row)

        df = pd.DataFrame(rows, index=index, columns=group_names)
        df.columns = pd.MultiIndex.from_tuples(
            [(self.groupby or "", c) for c in df.columns]
        )
        return df

    # ------------------------------------------------------------------ #
    # Display
    # ------------------------------------------------------------------ #
    def __repr__(self) -> str:
        return self.table.to_string()

    def _repr_html_(self) -> str:
        return self.table.to_html()

    def to_csv(self, path, **kwargs):
        return self.table.to_csv(path, **kwargs)

    def to_latex(self, **kwargs):
        return self.table.to_latex(**kwargs)

In [ ]:
dataset_path = "/opt/gpudata/ecg/heedb"
splits = ["train", "val", "test"]
data_kwargs = {
    "heedb_split_type": "by-label",
    "drop_blank_overread": True,
}
# ground truth labels: "original_physician" or "audited_physician" (has unknowns)
label_src = "audited_physician"
# original_physician keeps the original file name
csv_suffix = "" if label_src == "original_physician" else f"-{label_src}"
output_csv = f"temp_heedb_by_label_drop_blank{csv_suffix}.csv"

In [ ]:
dfs = []
for split in splits:
    ds = HeedbECGDataset(
        dataset_path=dataset_path,
        split=split,
        sampling_rate=100,
        label_src=label_src,
        **data_kwargs,
    )
    split_df = ds._df[["patient_id", "age", "sex", "source", "split"]].copy()
    # 1 / 0 / UNKNOWN_LABEL; only 1 is shown in the table
    split_df[list(HEEDB_TARGETS)] = ds.labels.numpy()
    dfs.append(split_df)

df = pd.concat(dfs, ignore_index=True)
df["sex"] = df["sex"].str.lower()
df.columns = ["Patient"] + list(df.columns.str.title())[1:]

tab1 = TableOne(
    data=df,
    columns=list(df.columns),
    nunique=["Patient"],
    categorical=[c for c in df.columns if c != "Age" and c != "Patient"],
    continuous=["Age"],
    nonnormal=["Age"],
    groupby="Split",
    order={
        "Split": list(df["Split"].drop_duplicates()),
    },
    binary_show={
        k.title(): 1 for k in HEEDB_TARGETS
    },
)

In [ ]:
tab1.to_csv(output_csv)
print(tab1.to_latex())

In [ ]:
tab1

In [ ]:
# PNG/PDF of the table for sharing, next to output_csv
from _table_figure import save_table

unknown_note = " Unknown labels count as not positive." if label_src == "audited_physician" else ""
save_table(
    tab1.table,
    output_csv,
    title=f"HEEDB by-label splits, blank overreads dropped ({label_src})",
    note=f"Label rows: ECGs positive in the physician overread, n (% of split).{unknown_note}",
)